# Proposed validation — review before it runs

**What this measures:** The target directly measures the Apriori miner's core mechanism — its ability to recover known deterministic cross-column implications (e.g. country⇒currency) from a synthetic reference frame at fixed support/confidence thresholds — which is the exact capability this PR adds as `mine_association_rules`. The guardrail checks that repeated calls with identical inputs yield an identical rule set, protecting the PR's central auditability claim (parameter-free, deterministic, no run-to-run drift) that must already trivially hold pre-change (absent feature ⇒ empty, still equal to itself).

**Target metric:** `cross_column_rule_recall`

Remyx wrote this test for the change in this PR. **Nothing here has been executed** — there are no outputs, and no result is being claimed.

Edit it if the measurement is wrong, then mention `@remyx validate` again and it will run what you committed. If anything is missing at run time — an import, a dependency, a device — the run reports it and repairs what it can rather than failing silently.

The executable copy lives at `eval/eval_cross_column_rules.py`, which is what `.remyx/validation.yaml` points at; keep the two in step, or point `suite:` here if you would rather maintain the notebook.

In [ ]:
"""Evaluation: cross-column association-rule mining (Apriori) recall + determinism.

Builds a synthetic 600-row reference DataFrame with 5 planted deterministic
single-column implications (country -> currency x3, plan -> tier x2) plus a
high-cardinality noise column that the miner must skip. Measures:

- cross_column_rule_recall: fraction of the 5 planted implications recovered
  by dataframe_expectations.expectations.cross_column_rules.mine_association_rules
  with method="apriori".
- cross_column_mining_determinism_rate: whether two mining calls on identical
  input produce an identical rule set (Apriori has no RNG / training loop).

In [ ]:
On baseline (changed module unimportable), mining falls back to an empty list
on both calls, so recall degrades to 0.0 while determinism trivially stays 1.0
(both empty results match) — the guardrail can never fail on baseline.
"""

import argparse
import json
import os
import sys

In [ ]:
import numpy as np
import pandas as pd

# Put repo root on sys.path so the real package is importable.
sys.path.insert(0, os.path.dirname(os.path.dirname(os.path.abspath(__file__))))

try:
    from dataframe_expectations.expectations.cross_column_rules import (
        mine_association_rules as _mine_association_rules,
    )

    HAVE_FEATURE = True
except Exception:
    HAVE_FEATURE = False

    def _mine_association_rules(*args, **kwargs):
        return []

In [ ]:
def build_reference_frame(rows: int = 600, seed: int = 0) -> pd.DataFrame:
    """600-row frame with 5 planted deterministic implications + noise column."""
    rng = np.random.default_rng(seed)
    country_to_currency = {"US": "USD", "DE": "EUR", "JP": "JPY"}
    plan_to_tier = {"basic": "bronze", "premium": "gold"}

    country_col = rng.choice(list(country_to_currency.keys()), size=rows)
    plan_col = rng.choice(list(plan_to_tier.keys()), size=rows)
    currency_col = [country_to_currency[c] for c in country_col]
    tier_col = [plan_to_tier[p] for p in plan_col]
    # High-cardinality identifier column the miner must skip (max_cardinality=50).
    noise_col = rng.integers(0, 1000, size=rows)

    return pd.DataFrame(
        {
            "country": country_col,
            "currency": currency_col,
            "plan": plan_col,
            "tier": tier_col,
            "noise_id": noise_col,
        }
    )

In [ ]:
def mine(data_frame: pd.DataFrame):
    """Call the real mine_association_rules with method='apriori' explicitly."""
    return _mine_association_rules(
        data_frame,
        min_support=0.05,
        min_confidence=0.9,
        max_antecedent_size=2,
        max_cardinality=50,
        method="apriori",
    )

In [ ]:
def rule_key(rule):
    """Hashable (antecedent, consequent) anchor for a mined rule."""
    antecedent = rule.antecedent if hasattr(rule, "antecedent") else rule[0]
    consequent = rule.consequent if hasattr(rule, "consequent") else rule[1]
    return (frozenset(antecedent.items()), frozenset(consequent.items()))

def main() -> None:
    parser = argparse.ArgumentParser()
    parser.add_argument("--variant", default=None)
    parser.add_argument("--ref", default=None)
    parser.add_argument("--seed", default=None)
    parser.parse_args()

    data_frame = build_reference_frame()

    planted = [
        ({"country": "US"}, {"currency": "USD"}),
        ({"country": "DE"}, {"currency": "EUR"}),
        ({"country": "JP"}, {"currency": "JPY"}),
        ({"plan": "basic"}, {"tier": "bronze"}),
        ({"plan": "premium"}, {"tier": "gold"}),
    ]
    planted_keys = {
        (frozenset(a.items()), frozenset(c.items())) for a, c in planted
    }

    try:
        rules_run1 = mine(data_frame)
    except Exception:
        rules_run1 = []
    try:
        rules_run2 = mine(data_frame)
    except Exception:
        rules_run2 = []

    mined_keys = set()
    for rule in rules_run1:
        try:
            mined_keys.add(rule_key(rule))
        except Exception:
            continue

    recovered = sum(1 for key in planted_keys if key in mined_keys)
    recall = recovered / len(planted_keys)

    try:
        keys1 = sorted(str(rule_key(rule)) for rule in rules_run1)
        keys2 = sorted(str(rule_key(rule)) for rule in rules_run2)
        determinism_rate = 1.0 if keys1 == keys2 else 0.0
    except Exception:
        determinism_rate = 0.0

    if not HAVE_FEATURE:
        recall = 0.0
        determinism_rate = 1.0  # both calls trivially return [] == []

    print(
        json.dumps(
            {
                "cross_column_rule_recall": recall,
                "cross_column_mining_determinism_rate": determinism_rate,
            }
        )
    )

if __name__ == "__main__":
    main()

## The criteria this is judged against

From `.remyx/validation.yaml` — thresholds live here, not in the test, so a failing measurement reports rather than crashes.

```yaml
benchmarks:
  - suite: "eval/eval_cross_column_rules.py"
    scorer: cross_column_rule_recall
    baseline: main
    metrics:
      - key: cross_column_rule_recall
        role: target
        direction: max
        # 5 planted deterministic implications (country->currency x3, plan->tier x2);
        # Apriori with min_support=0.05/min_confidence=0.9 should recover all 5 (recall=1.0)
        # since every antecedent value maps to exactly one consequent value in the
        # synthetic reference frame. Baseline (feature absent) recalls 0. 0.8 leaves
        # margin for enumeration-order edge cases while still separating the arms.
        threshold: 0.8
      - key: cross_column_mining_determinism_rate
        role: guardrail
        direction: max
        # Fraction of mined rule-sets identical across two calls with identical
        # inputs. Apriori has no training loop/RNG, so this must be exactly 1.0;
        # baseline's fallback (empty list both times) is trivially 1.0 too, so the
        # guardrail cannot fail on baseline but does fail if mining becomes
        # nondeterministic (e.g. an unseeded stochastic default).
        threshold: 1.0
    held_constant:
      - "same synthetic reference DataFrame (600 rows, 2 planted deterministic implication families) across both arms"
      - "same min_support=0.05 / min_confidence=0.9 / max_antecedent_size / max_cardinality thresholds"
      - "same method='apriori' argument passed explicitly on both arms"
    avoid:
      - "no network access"
      - "no wall-clock timing (mining cost is not the claim; correctness/determinism is)"
      - "no real pyspark/polars import required — only pandas, mirroring the module's own lazy-proxy import pattern"
    compute:
      tier: cpu
    provenance:
      cross_column_rule_recall: "synthesized from PR source dataframe_expectations/expectations/cross_column_rules.py (mine_association_rules apriori docstring: 'parameter-free frequency/confidence miner ... deterministic')"
      cross_column_mining_determinism_rate: "synthesized from PR description: 'Deterministic Apriori miner as the default ... no hyperparameters that shift the rule set between runs'"
      held_constant: "synthesized from PR docstrings describing mine_association_rules parameters"
      suite: "synthesized, mirrors tests/expectations/test_cross_column_rules.py style (direct calls to mine_association_rules)"
    policy:
      guardrail_veto: true
```